# Semaine 5 — Jour 5 : Monitoring des systèmes IA en production

## Position dans le bootcamp

La semaine 5 construit les fondations d’un système IA exploitable en production : architecture, API, cache Redis, persistance PostgreSQL, monitoring, sécurité et déploiement.

Ce jour 5 est consacré au **monitoring**. L’objectif n’est pas seulement de vérifier qu’une API répond. Un système IA en production doit être observable sur plusieurs dimensions :

- disponibilité de l’API ;
- latence bout-en-bout ;
- erreurs techniques ;
- erreurs fonctionnelles ;
- consommation de tokens ;
- coût estimé ;
- qualité des réponses ;
- dérive de comportement ;
- saturation des dépendances ;
- incidents exploitables par une équipe.

## Problème traité

Une application IA peut paraître saine au niveau HTTP tout en étant dégradée côté métier.

Exemples :

- le modèle répond toujours `200 OK`, mais hallucine davantage ;
- les coûts doublent à cause de prompts trop longs ;
- la latence augmente uniquement pour certains types de requêtes ;
- une dépendance externe ralentit sans provoquer d’erreur explicite ;
- les réponses refusées ou vides augmentent progressivement ;
- un agent boucle trop longtemps avant de produire une réponse.

Le monitoring d’un système IA doit donc combiner **observabilité logicielle classique** et **métriques spécifiques aux LLM**.

## Livrables de la journée

À la fin de la journée, l’apprenant sait concevoir et implémenter un socle minimal de monitoring pour une API IA :

- métriques techniques ;
- métriques IA ;
- endpoint `/metrics` ;
- endpoint `/health` ;
- règles d’alerte ;
- estimation de coûts ;
- journalisation structurée ;
- redaction de données sensibles ;
- tests automatisés sur la logique de monitoring.

## Lab

Le lab construit une mini-API FastAPI instrumentée :

```text
labs/monitoring_ai_service/
├── app.py
├── metrics.py
├── alerts.py
├── cost.py
├── redaction.py
├── schemas.py
└── tests/
```

Le lab n’appelle pas de vrai modèle externe. Il simule une réponse IA afin que le monitoring reste reproductible, testable et exécutable localement.

## Pré-requis

- Python 3.11+
- bases de FastAPI
- bases de Redis et PostgreSQL vues les jours précédents
- notions de latence, erreurs HTTP et logs applicatifs

## Résultat attendu

L’apprenant doit être capable d’expliquer, coder et tester un système de monitoring minimal pour une API IA en production.

## Parcours conseillé — 60 minutes

1. Objectifs et chapitre : 20 minutes.
2. Installation et lab : 20 minutes.
3. Exercices et diagnostic : 15 minutes.
4. Correction et questions d'entretien : 5 minutes.

Le challenge constitue un approfondissement facultatif. Commencer par
`learning_objectives.md`, puis `chapter.md` et `labs/README.md`.

## Intégration au dépôt

Copier les fichiers de cette archive en conservant les autres jours. Aucun fichier
`__init__.py` partagé du mini-framework n'est remplacé. Les notebooks de la journée
sont fournis dans `notebooks/week05/`. Le lab est autonome ; son raccordement aux
services réels des jours 2 à 4 reste un travail d'intégration explicite.


# Objectifs pédagogiques

## Objectifs principaux

À la fin de cette journée, l’apprenant sera capable de :

1. distinguer logging, métriques, traces et alerting ;
2. identifier les métriques spécifiques aux systèmes IA ;
3. instrumenter une API FastAPI avec des métriques de latence, d’erreurs et de volume ;
4. exposer un endpoint `/metrics` lisible par une plateforme de supervision ;
5. calculer une estimation de coût basée sur les tokens ;
6. détecter des conditions d’alerte simples ;
7. protéger les logs contre l’exposition de données sensibles ;
8. tester la logique de monitoring sans dépendre d’un fournisseur LLM.

## Compétences AI Engineering

Cette journée développe les compétences suivantes :

- concevoir un contrat d’observabilité pour une API IA ;
- rendre un service IA exploitable par une équipe production ;
- séparer les métriques système des métriques modèle ;
- transformer des événements applicatifs en signaux actionnables ;
- écrire du code de monitoring testable ;
- raisonner en coût, latence, fiabilité et qualité.

## Notions clés

- Observabilité
- Logging structuré
- Métriques
- Traces
- SLO
- Error rate
- P95 latency
- Token usage
- Cost monitoring
- Model quality signal
- Alerting
- Redaction
- Runbook

## Critères de réussite

La journée est réussie si l’apprenant peut :

- ajouter une métrique sans modifier toute l’API ;
- expliquer pourquoi `200 OK` ne suffit pas à mesurer la santé d’un système IA ;
- calculer une estimation de coût par requête ;
- écrire une règle d’alerte compréhensible ;
- filtrer les données sensibles avant journalisation ;
- exécuter les tests du lab avec succès.


# Chapitre — Monitoring des systèmes IA en production

## 1. Pourquoi le monitoring IA est différent

Le monitoring d’une API classique répond souvent à trois questions :

1. le service est-il disponible ?
2. répond-il assez vite ?
3. produit-il trop d’erreurs ?

Ces questions restent nécessaires pour une API IA, mais elles sont insuffisantes. Une API IA peut répondre rapidement et sans erreur HTTP tout en produisant une réponse inutilisable.

Un système IA introduit plusieurs sources de risque supplémentaires :

- coût variable selon les tokens consommés ;
- latence variable selon la taille du prompt et du modèle ;
- qualité difficile à mesurer automatiquement ;
- dépendance à des fournisseurs externes ;
- comportement probabiliste ;
- risques de fuite de données dans les prompts et logs ;
- erreurs métier non visibles dans les codes HTTP.

Le monitoring IA doit donc enrichir l’observabilité classique avec des signaux métier et modèle.

## 2. Les quatre piliers opérationnels

Dans un système IA en production, quatre familles de signaux sont particulièrement importantes.

### 2.1 Logs

Les logs décrivent des événements.

Exemple :

```json
{
  "event": "ai_request_completed",
  "request_id": "req_123",
  "route": "/chat",
  "status_code": 200,
  "latency_ms": 842,
  "model": "gpt-example",
  "prompt_tokens": 312,
  "completion_tokens": 84,
  "estimated_cost_usd": 0.0048
}
```

Un bon log est structuré, filtré, corrélable et exploitable.

Un mauvais log contient une chaîne de texte impossible à agréger ou expose directement des informations sensibles.

### 2.2 Métriques

Les métriques agrègent des mesures dans le temps.

Exemples :

- nombre de requêtes ;
- taux d’erreur ;
- latence moyenne ;
- latence P95 ;
- tokens consommés ;
- coût estimé ;
- nombre de refus ;
- nombre de réponses vides ;
- nombre de dépassements de budget.

Les métriques doivent être peu coûteuses à produire et faciles à agréger.

### 2.3 Traces

Les traces décrivent le parcours d’une requête à travers plusieurs composants.

Dans un système IA, une requête peut traverser :

1. l’API ;
2. un cache Redis ;
3. une base PostgreSQL ;
4. un orchestrateur d’agent ;
5. un outil externe ;
6. un fournisseur LLM ;
7. un composant de post-traitement.

Sans trace, il devient difficile d’identifier l’étape responsable d’une lenteur.

### 2.4 Alertes

Les alertes transforment des métriques en actions.

Une bonne alerte doit être :

- actionnable ;
- compréhensible ;
- liée à un seuil explicite ;
- associée à un runbook ;
- limitée pour éviter la fatigue d’alerte.

Une mauvaise alerte se déclenche trop souvent ou ne dit pas quoi faire.

## 3. Architecture de monitoring

Le schéma suivant montre une architecture minimale de monitoring pour une API IA.

```mermaid
flowchart TD
    U[Utilisateur] --> API[FastAPI AI Service]
    API --> C[Cache Redis]
    API --> DB[(PostgreSQL)]
    API --> LLM[Fournisseur LLM]
    API --> LOGS[Logs structurés]
    API --> METRICS[Endpoint /metrics]
    API --> HEALTH[Endpoint /health]
    METRICS --> DASH[Dashboard]
    LOGS --> DASH
    DASH --> ALERTS[Alerting]
    ALERTS --> OPS[Equipe Ops / AI Engineering]
```

Cette architecture sépare les responsabilités :

- l’API produit les signaux ;
- les dashboards les rendent lisibles ;
- les alertes déclenchent une action ;
- les runbooks guident l’intervention.

## 4. Métriques techniques indispensables

Une API IA doit au minimum suivre :

| Métrique | Description | Pourquoi c’est important |
|---|---|---|
| `request_count` | nombre total de requêtes | mesure le trafic |
| `error_count` | nombre d’erreurs | détecte les incidents |
| `error_rate` | erreurs / requêtes | signale une dégradation |
| `avg_latency_ms` | latence moyenne | suit la performance globale |
| `p95_latency_ms` | 95e percentile de latence | détecte les lenteurs utilisateur |
| `status_code_count` | distribution des codes HTTP | identifie les erreurs dominantes |
| `dependency_latency_ms` | latence des dépendances | localise les goulots |
| `uptime_seconds` | durée depuis le démarrage | aide au diagnostic |

La latence P95 est souvent plus utile que la moyenne, car elle reflète mieux l’expérience des utilisateurs les plus impactés.

## 5. Métriques spécifiques IA

Les métriques IA complètent les métriques techniques.

| Métrique | Description | Exemple d’usage |
|---|---|---|
| `prompt_tokens` | tokens en entrée | détecter les prompts trop longs |
| `completion_tokens` | tokens en sortie | suivre la verbosité |
| `total_tokens` | somme entrée + sortie | piloter le coût |
| `estimated_cost_usd` | coût estimé | surveiller le budget |
| `model_name` | modèle utilisé | comparer les performances |
| `cache_hit` | réponse servie depuis cache | mesurer l’efficacité Redis |
| `empty_response_count` | réponses vides | détecter un problème fonctionnel |
| `refusal_count` | refus ou blocages | analyser les limites produit |
| `quality_score` | score interne ou évaluation | suivre la qualité |
| `tool_call_count` | nombre d’appels d’outils | détecter les agents trop bavards |

Ces métriques doivent être collectées sans stocker inutilement les prompts bruts.

## 6. Les SLO d’un système IA

Un SLO, ou Service Level Objective, définit un objectif mesurable.

Exemples :

- 99 % des requêtes `/chat` répondent en moins de 3 secondes ;
- le taux d’erreur reste inférieur à 1 % sur 15 minutes ;
- le coût journalier reste inférieur à 50 dollars ;
- moins de 2 % des réponses sont vides ;
- le cache Redis sert au moins 30 % des requêtes répétées.

Un SLO n’est pas une promesse abstraite. C’est un contrat exploitable qui guide le monitoring, les alertes et les priorités techniques.

## 7. Exemple de contrat de métriques

Un endpoint `/metrics` minimal peut retourner un document JSON :

```json
{
  "request_count": 1200,
  "error_count": 18,
  "error_rate": 0.015,
  "avg_latency_ms": 812.4,
  "p95_latency_ms": 1840.0,
  "total_prompt_tokens": 210000,
  "total_completion_tokens": 72000,
  "estimated_cost_usd": 4.92
}
```

Dans un environnement réel, ces métriques seraient souvent exposées au format Prometheus. Dans ce bootcamp, le JSON est suffisant pour comprendre et tester les principes.

## 8. Corrélation par request id

Chaque requête doit recevoir un identifiant unique.

Cet identifiant est propagé dans :

- les logs ;
- les enregistrements internes du lab ;
- les traces ;
- les erreurs retournées au client ;
- les appels aux dépendances.

Exemple :

```text
request_id=req_01HZX route=/chat latency_ms=642 model=gpt-example status=200
```

Sans identifiant de corrélation, le diagnostic devient difficile dès que plusieurs utilisateurs sollicitent le système en même temps.

## 9. Monitoring du coût

Le coût d’une API IA n’est pas fixe. Il dépend généralement :

- du modèle utilisé ;
- des tokens d’entrée ;
- des tokens de sortie ;
- des appels d’outils ;
- des retries ;
- du cache ;
- du volume utilisateur.

Une estimation simple peut être formulée ainsi :

```text
coût = prompt_tokens / 1000 * prix_input
     + completion_tokens / 1000 * prix_output
```

Même approximative, cette mesure est utile pour :

- détecter une régression de prompt ;
- identifier un utilisateur ou cas d’usage coûteux ;
- déclencher une alerte de budget ;
- comparer plusieurs architectures.

## 10. Monitoring de la qualité

La qualité est la partie la plus difficile.

Une réponse IA peut être :

- techniquement valide ;
- grammaticalement correcte ;
- mais fausse, incomplète ou dangereuse.

Pour un premier système, on peut commencer avec des signaux simples :

- réponse vide ;
- réponse trop courte ;
- réponse trop longue ;
- présence d’une phrase de fallback ;
- score d’un évaluateur automatique ;
- feedback utilisateur ;
- divergence avec une réponse attendue ;
- nombre de reformulations nécessaires.

Ces signaux ne remplacent pas une évaluation robuste, mais ils permettent de détecter rapidement une dégradation.

## 11. Redaction des données sensibles

Le monitoring ne doit pas devenir une fuite de données.

Avant de logger un prompt, une sortie ou un message d’erreur, il faut filtrer :

- emails ;
- numéros de téléphone ;
- clés API ;
- tokens d’accès ;
- secrets ;
- informations personnelles ;
- contenus client confidentiels.

Dans beaucoup de systèmes, le choix le plus sûr consiste à ne pas logger le prompt complet. On conserve plutôt :

- longueur ;
- hash ;
- type de tâche ;
- langue ;
- utilisateur ou tenant pseudonymisé ;
- nombre de tokens.

## 12. Alerting minimal

Une règle d’alerte doit combiner une condition et une action.

Exemple :

```yaml
name: high_error_rate
condition: error_rate > 0.05
severity: critical
runbook: check_recent_deployments_and_provider_status
```

Les premières règles utiles sont :

- taux d’erreur élevé ;
- P95 trop élevé ;
- coût estimé trop élevé ;
- trop de réponses vides ;
- dépendance indisponible ;
- absence de trafic attendu.

## 13. Runbook

Un runbook est une procédure de diagnostic.

Exemple pour une hausse de latence :

1. vérifier si la hausse concerne toutes les routes ;
2. comparer latence API et latence fournisseur LLM ;
3. vérifier les appels Redis et PostgreSQL ;
4. inspecter les prompts récents ;
5. vérifier les retries ;
6. regarder les changements de déploiement ;
7. appliquer mitigation : cache, rollback, réduction de contexte, fallback modèle.

Le runbook transforme une alerte en action opérationnelle.

## 14. Patterns de conception

Un bon monitoring doit être découplé du code métier.

Mauvais pattern :

```python
def chat(prompt):
    start = time.time()
    response = call_model(prompt)
    print("latency", time.time() - start)
    return response
```

Meilleur pattern :

```python
def chat(prompt, monitor):
    with monitor.track("chat"):
        response = call_model(prompt)
        monitor.record_tokens(prompt_tokens=120, completion_tokens=60)
        return response
```

L’objectif est de rendre la collecte testable, réutilisable et évolutive.

## 15. Exemple de cycle production

```mermaid
sequenceDiagram
    participant Client
    participant API
    participant Monitor
    participant Model
    participant Dashboard

    Client->>API: POST /chat
    API->>Monitor: start request span
    API->>Model: generate response
    Model-->>API: response + token usage
    API->>Monitor: record latency, status, tokens, cost
    API-->>Client: answer + request_id
    Dashboard->>Monitor: GET /metrics
    Dashboard->>Monitor: evaluate thresholds
```

Ce cycle montre que le monitoring n’est pas une tâche secondaire. Il fait partie du chemin de production.

## 16. Ce qu’il faut éviter

À éviter :

- logger les prompts complets sans filtrage ;
- confondre uptime et qualité ;
- surveiller uniquement la moyenne de latence ;
- déclencher des alertes sans runbook ;
- ignorer les coûts ;
- ajouter des métriques non utilisées ;
- mélanger métriques techniques et données personnelles ;
- monitorer uniquement le fournisseur LLM ;
- ne pas tester la logique de monitoring.

## 17. Synthèse

Un système IA production-ready doit être observable.

Le monitoring répond à trois questions opérationnelles :

1. le système fonctionne-t-il ?
2. fonctionne-t-il bien ?
3. fonctionne-t-il à un coût et un niveau de risque acceptables ?

Pour un AI Backend Engineer ou AI Solution Architect, le monitoring est une compétence centrale. Il permet de passer d’une démonstration IA à un service fiable, exploitable et améliorable.


## 18. Du lab vers la production : limites à comprendre

Le lab expose `/metrics` en JSON : Prometheus ne peut pas collecter directement
ce contrat. Son adaptation nécessite un client d'instrumentation et un endpoint
au format d'exposition Prometheus. Les règles YAML fournies sont pédagogiques,
pas des règles Prometheus exécutables. `/alerts` évalue les seuils seulement
lorsqu'il est appelé et ne transmet aucune notification.

Les compteurs, coûts et percentiles sont calculés depuis le démarrage du processus.
Ils ne représentent ni une fenêtre de quinze minutes ni un budget journalier.
Une alerte de coût cumulatif reste donc active après le franchissement du seuil.
Pour un SLO, préciser la fenêtre, les requêtes éligibles et le volume minimal avant
une alerte. Le P95 local ne doit jamais être moyenné entre plusieurs instances.

Le collecteur conserve tous les événements en mémoire : le lab s'exécute avec un
seul worker et une charge limitée. En production, utiliser des compteurs et des
histogrammes exportés vers un stockage de séries temporelles. Garder `request_id`
dans les logs et traces, jamais dans les labels de métriques : chaque identifiant
créerait une nouvelle série. Les labels route, modèle et statut doivent avoir un
nombre de valeurs borné.

`/health` vérifie ici un service simulé. Il ne teste pas Redis, PostgreSQL ou un
fournisseur LLM réel. Distinguer liveness (processus vivant) et readiness (capacité
à servir). Les métriques de refus, cache et qualité présentées dans le chapitre
sont des pistes d'extension, pas des mesures déjà implémentées.

Le nombre de tokens est une approximation par caractères et les prix sont des
hypothèses pédagogiques. Utiliser les compteurs de consommation du fournisseur
et ses tarifs configurés pour la facturation réelle. Une réponse vide est un
signal de qualité ; elle ne mesure pas la factualité. Les expressions régulières
de masquage ne garantissent pas une anonymisation exhaustive. Le middleware
journalise des métadonnées sans enregistrer le contenu utilisateur.


# Exercices

## Exercice 1 — Identifier les métriques

Pour chaque situation, indique la métrique principale à surveiller.

1. Les utilisateurs se plaignent que l’API répond lentement.
2. La facture LLM augmente soudainement.
3. Le service répond `200 OK`, mais produit parfois une réponse vide.
4. Les erreurs augmentent après un déploiement.
5. Les prompts envoyés au modèle deviennent plus longs.

## Exercice 2 — Calculer un coût estimé

On utilise le barème fictif suivant :

```text
input: 0.002 USD / 1000 tokens
output: 0.006 USD / 1000 tokens
```

Une requête consomme :

```text
prompt_tokens = 1500
completion_tokens = 400
```

Calcule le coût estimé de la requête.

## Exercice 3 — Définir des SLO

Propose trois SLO pour une API `/chat` utilisée en production :

- un SLO de latence ;
- un SLO d’erreur ;
- un SLO de coût.

Chaque SLO doit être mesurable.

## Exercice 4 — Redaction

On veut logger le message suivant :

```text
Bonjour, mon email est alice@example.com et ma clé API est sk-live-1234567890abcdef.
```

Propose une version filtrée adaptée aux logs.

## Exercice 5 — Lire un snapshot de métriques

On observe le snapshot suivant :

```json
{
  "request_count": 1000,
  "error_count": 70,
  "error_rate": 0.07,
  "avg_latency_ms": 900,
  "p95_latency_ms": 4200,
  "estimated_cost_usd": 83.5,
  "empty_response_count": 18
}
```

Hypothèses :

- seuil erreur critique : `error_rate > 0.05`
- seuil P95 critique : `p95_latency_ms > 3000`
- seuil coût : `estimated_cost_usd > 75`

Quelles alertes doivent se déclencher ?

## Exercice 6 — Instrumentation

Écris une fonction Python `record_ai_request` qui reçoit :

- `metrics_store`
- `latency_ms`
- `status_code`
- `prompt_tokens`
- `completion_tokens`
- `estimated_cost_usd`
- `empty_response`

La fonction doit enregistrer une requête dans `metrics_store`.

Tu peux supposer que `metrics_store` expose une méthode `record(...)`.

## Exercice 7 — Qualité IA

Donne quatre signaux simples qui permettent de détecter une dégradation de qualité dans une application IA.


# Questions d’entretien

## Question 1

Pourquoi le taux de disponibilité HTTP ne suffit-il pas pour monitorer une application IA ?

## Question 2

Quelle est la différence entre logs, métriques et traces ?

## Question 3

Pourquoi la latence P95 est-elle souvent plus importante que la latence moyenne ?

## Question 4

Comment monitorer le coût d’un système basé sur des LLM ?

## Question 5

Quelles données ne faut-il pas logger dans une application IA ?

## Question 6

Comment concevoir une alerte utile ?

## Question 7

Comment diagnostiquer une hausse soudaine de latence dans un agent IA ?

## Question 8

Pourquoi tester le code de monitoring ?


# Challenge — Construire un monitoring minimal pour une API IA

## Contexte

Tu travailles sur une API `/chat` utilisée par une équipe produit. L’API appelle un modèle IA et retourne une réponse à l’utilisateur.

Depuis quelques jours, l’équipe observe trois problèmes :

1. certains utilisateurs se plaignent de lenteurs ;
2. la facture LLM augmente ;
3. quelques réponses sont vides, mais l’API retourne quand même `200 OK`.

Tu dois concevoir un monitoring minimal pour rendre ces problèmes visibles.

## Objectif

Créer un système de monitoring applicatif capable de :

- enregistrer chaque requête IA ;
- mesurer latence, statut HTTP, tokens, coût et réponses vides ;
- exposer un snapshot de métriques ;
- déclencher des alertes simples ;
- éviter de logger des secrets.

## Contraintes

- Le système doit être testable sans fournisseur LLM réel.
- Le stockage en mémoire est accepté pour le challenge.
- Les prompts bruts ne doivent pas être conservés dans les métriques.
- Les règles d’alerte doivent être lisibles.
- Le code doit être séparé en composants.

## Attendus

Le rendu doit inclure :

1. un modèle de données pour représenter une requête monitorée ;
2. un collecteur de métriques ;
3. une fonction de calcul de coût ;
4. au moins trois règles d’alerte ;
5. une fonction de redaction ;
6. des tests unitaires ;
7. une courte explication de l’architecture.

## Bonus

Ajouter :

- un endpoint `/health` ;
- un endpoint `/metrics` ;
- une route `/chat` simulée ;
- un `request_id` retourné au client ;
- un runbook associé à chaque alerte.


# Lab — Monitoring d’une API IA

## Objectif

Construire une mini-API FastAPI instrumentée pour exposer :

- `/health`
- `/metrics`
- `/alerts`
- `/chat`

Le fournisseur LLM est simulé pour rendre le lab reproductible.

## Installation

Depuis ce dossier :

```bash
python -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
```

## Lancer les tests

```bash
python -m pytest -q
```

## Lancer l’API

```bash
python -m uvicorn monitoring_ai_service.app:app --reload
```

## Tester manuellement

```bash
curl -X POST http://127.0.0.1:8000/chat \
  -H "Content-Type: application/json" \
  -d '{"message": "Explique le monitoring IA en trois phrases."}'
```


In [ ]:
from pathlib import Path
import sys
roots = [Path.cwd(), *Path.cwd().parents]
lab = next((r / 'book/week05/day05/labs' for r in roots if (r / 'book/week05/day05/labs').is_dir()), None)
if lab is None:
    raise RuntimeError('Ouvrir ce notebook depuis le dépôt ai-engineering-bootcamp.')
sys.path.insert(0, str(lab))
from fastapi.testclient import TestClient
from monitoring_ai_service.app import app, metrics_store
metrics_store.reset()
client = TestClient(app)
assert client.post('/chat', json={'message': 'Bonjour'}).status_code == 200
assert client.get('/metrics').json()['request_count'] == 1
client.get('/metrics').json()
